# 02 — Model Eğitimi
**Giriş:** `packed_landmarks/train_X.npy` → shape `(N, 16, 450)`

**Mimari:** Transformer Encoder + Classification Head
- Positional Encoding
- 4× Multi-Head Attention bloğu
- GlobalAvgPool
- Dense head → 226 sınıf

**Teknikler:**
- Data augmentation (noise, flip, time warp)
- Class weight (dengesiz dataset için)
- Label smoothing
- Cosine LR decay
- Mixed precision eğitim

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os, json, time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, mixed_precision
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import top_k_accuracy_score

print("TF:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

mixed_precision.set_global_policy('mixed_float16')

TF: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [6]:
# ── CONFIG ────────────────────────────────────────────────
BASE         = "/content/drive/MyDrive/sign-language-project"
PACKED_DIR   = f"{BASE}/packed_landmarks"
SIGN_CSV     = f"{BASE}/dataset/SignList_ClassId_TR_EN.csv"
SAVE_PATH    = f"{BASE}/transformer_226_best.keras"
HISTORY_PATH = f"{BASE}/training_history.json"

SEQ_LEN      = 16
FEAT_DIM     = 252
NUM_CLASSES  = 226
BATCH_SIZE   = 64
EPOCHS       = 100
PATIENCE     = 15       # early stopping

In [7]:
# ── VERİ YÜKLE ────────────────────────────────────────────
X_train = np.load(f"{PACKED_DIR}/train_X.npy")
y_train = np.load(f"{PACKED_DIR}/train_y.npy")
X_val   = np.load(f"{PACKED_DIR}/val_X.npy")
y_val   = np.load(f"{PACKED_DIR}/val_y.npy")

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape},   y_val:   {y_val.shape}")
print(f"Unique sınıf (train): {len(np.unique(y_train))}")
print(f"Label aralığı: {y_train.min()} - {y_train.max()}")

# NaN/Inf kontrolü
assert not np.isnan(X_train).any(), "NaN var!"
assert not np.isinf(X_train).any(), "Inf var!"
print("✅ Veri temiz")

X_train: (28142, 16, 252), y_train: (28142,)
X_val:   (4418, 16, 252),   y_val:   (4418,)
Unique sınıf (train): 226
Label aralığı: 0 - 225
✅ Veri temiz


In [8]:
# ── NORMALİZASYON ─────────────────────────────────────────
# Landmark koordinatları zaten 0-1 arasında ama
# sıfır (eksik landmark) ayrımını korumak için
# sadece nonzero elementleri normalize ediyoruz

# Per-feature mean/std (train üzerinden)
# Sıfırları hariç tutarak istatistik hesapla
mask = X_train != 0
feat_mean = np.zeros(FEAT_DIM, dtype=np.float32)
feat_std  = np.ones(FEAT_DIM, dtype=np.float32)

for d in range(FEAT_DIM):
    vals = X_train[:, :, d][mask[:, :, d]]
    if len(vals) > 0:
        feat_mean[d] = vals.mean()
        feat_std[d]  = max(vals.std(), 1e-6)

# Normalize et (sıfırları sıfır bırak)
def normalize(X, mean, std):
    X_norm = X.copy()
    nonzero = X != 0
    X_norm[nonzero] = (X[nonzero] - mean[np.where(nonzero)[2]] ) / std[np.where(nonzero)[2]]
    return X_norm

# Daha basit ve hızlı versiyon: global normalize, sıfırlar yakın kalır
X_train_norm = (X_train - feat_mean) / feat_std
X_val_norm   = (X_val   - feat_mean) / feat_std

# Normalizasyon istatistiklerini kaydet (inference için lazım)
norm_stats = {'mean': feat_mean.tolist(), 'std': feat_std.tolist()}
with open(f"{BASE}/norm_stats.json", 'w') as f:
    json.dump(norm_stats, f)

print(f"Normalizasyon tamamlandı. mean range: [{feat_mean.min():.3f}, {feat_mean.max():.3f}]")
print("norm_stats.json kaydedildi")

Normalizasyon tamamlandı. mean range: [-0.033, 0.759]
norm_stats.json kaydedildi


In [11]:
# ── DATA AUGMENTATION ─────────────────────────────────────
@tf.function
def augment(x, y):
    """
    Eğitim sırasında rastgele augmentation uygular.
    """
    # 1) Gaussian noise
    if tf.random.uniform(()) > 0.5:
        noise = tf.random.normal(tf.shape(x), stddev=0.02)
        x = x + noise

    # 2) Horizontal flip (x koordinatlarını ters çevir)
    # Landmark'lar normalize edilmiş, x değerlerini 1-x yap
    # Her 3'te bir eleman x koordinatı (x,y,z sırası)
    if tf.random.uniform(()) > 0.5:
        # Basit yaklaşım: x = 1 - x olması gereken yerleri ters çevir
        # Feature'lar zaten normalize edildiği için sadece işaret değiştir
        x = x * tf.constant([-1 if i % 3 == 0 else 1 for i in range(252)], dtype=tf.float32)

    # 3) Time masking: 1-2 frame'i sıfırla
    if tf.random.uniform(()) > 0.7:
        mask_len = tf.random.uniform((), 1, 3, dtype=tf.int32)
        start    = tf.random.uniform((), 0, SEQ_LEN - mask_len, dtype=tf.int32)
        mask     = tf.concat([
            tf.ones([start, FEAT_DIM]),
            tf.zeros([mask_len, FEAT_DIM]),
            tf.ones([SEQ_LEN - start - mask_len, FEAT_DIM])
        ], axis=0)
        x = x * tf.cast(mask, tf.float32)

    # 4) Scale jitter
    if tf.random.uniform(()) > 0.5:
        scale = tf.random.uniform((), 0.9, 1.1)
        x = x * scale

    return x, y

print("Augmentation fonksiyonu hazır")

Augmentation fonksiyonu hazır


In [12]:
# ── tf.data PIPELINE ──────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

def make_dataset(X, y, training=False, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices(
        (X.astype(np.float32), y.astype(np.int32))
    )
    if training:
        ds = ds.shuffle(buffer_size=len(X), reshuffle_each_iteration=True)
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(X_train_norm, y_train, training=True)
val_ds   = make_dataset(X_val_norm,   y_val,   training=False)

print("Dataset pipeline hazır")

Dataset pipeline hazır


In [13]:
# ── MODEL MİMARİSİ ────────────────────────────────────────
def positional_encoding(seq_len, d_model):
    """Sinüzoidal positional encoding."""
    positions = np.arange(seq_len)[:, np.newaxis]
    dims      = np.arange(d_model)[np.newaxis, :]
    angles    = positions / np.power(10000, (2 * (dims // 2)) / d_model)
    angles[:, 0::2] = np.sin(angles[:, 0::2])
    angles[:, 1::2] = np.cos(angles[:, 1::2])
    return tf.cast(angles[np.newaxis, :, :], tf.float32)  # (1, seq_len, d_model)


def transformer_encoder_block(x, d_model, num_heads, ff_dim, dropout_rate):
    """Tek bir Transformer Encoder bloğu."""
    # Multi-Head Attention
    attn_out = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_model // num_heads,
        dropout=dropout_rate
    )(x, x)
    attn_out = layers.Dropout(dropout_rate)(attn_out)
    x = layers.LayerNormalization(epsilon=1e-6)(x + attn_out)

    # Feed-Forward Network
    ff_out = layers.Dense(ff_dim, activation='gelu')(x)
    ff_out = layers.Dropout(dropout_rate)(ff_out)
    ff_out = layers.Dense(d_model)(ff_out)
    ff_out = layers.Dropout(dropout_rate)(ff_out)
    x = layers.LayerNormalization(epsilon=1e-6)(x + ff_out)

    return x


def build_transformer_model(
    seq_len=16,
    feat_dim=450,
    num_classes=226,
    d_model=256,
    num_heads=8,
    ff_dim=512,
    num_blocks=4,
    dropout_rate=0.3,
):
    inputs = keras.Input(shape=(seq_len, feat_dim), name='landmark_seq')

    # Input projection
    x = layers.Dense(d_model, name='input_proj')(inputs)
    x = layers.LayerNormalization()(x)

    # Positional encoding ekle
    pos_enc = positional_encoding(seq_len, d_model)
    x = x + pos_enc
    x = layers.Dropout(dropout_rate)(x)

    # Transformer bloğları
    for i in range(num_blocks):
        x = transformer_encoder_block(x, d_model, num_heads, ff_dim, dropout_rate)

    # Aggregation: hem avg hem max pool, concat
    avg_pool = layers.GlobalAveragePooling1D()(x)
    max_pool = layers.GlobalMaxPooling1D()(x)
    x = layers.Concatenate()([avg_pool, max_pool])  # (d_model * 2,)

    # Classification head
    x = layers.Dense(512, activation='gelu')(x)
    x = layers.Dropout(dropout_rate)(x)
    x = layers.Dense(256, activation='gelu')(x)
    x = layers.Dropout(dropout_rate)(x)

    # Output (float32 — mixed precision için önemli)
    outputs = layers.Dense(num_classes, activation='softmax', dtype='float32', name='output')(x)

    model = keras.Model(inputs, outputs, name='TSL_Transformer')
    return model


model = build_transformer_model()
model.summary()
print(f"\nToplam parametre: {model.count_params():,}")

Model: "TSL_Transformer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ landmark_seq        │ (None, 16, 450)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_proj (Dense)  │ (None, 16, 256)   │    115,456 │ landmark_seq[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 16, 256)   │        512 │ input_proj[0][0]  │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 16, 256)   │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 16, 256)   │          0 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 16, 256)   │    263,168 │ dropout[0][0],    │
│ (MultiHeadAttentio… │                   │            │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 16, 256)   │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 16, 256)   │          0 │ dropout[0][0],    │
│                     │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 16, 256)   │        512 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 16, 512)   │    131,584 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 16, 512)   │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 16, 256)   │    131,328 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 16, 256)   │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 16, 256)   │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 16, 256)   │        512 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 16, 256)   │    263,168 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 16, 256)   │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 16, 256)   │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_6[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 16, 256)   │        512 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 16, 512)   │    131,584 │ layer_normalizat

 Total params: 2,676,450 (10.21 MB)

 Trainable params: 2,676,450 (10.21 MB)

 Non-trainable params: 0 (0.00 B)


Toplam parametre: 2,676,450


In [ ]:
# ── CLASS WEIGHT ──────────────────────────────────────────
classes = np.arange(NUM_CLASSES)
cw = compute_class_weight('balanced', classes=classes, y=y_train)

# Çok nadir sınıfları sınırla (eğitimi dengesizleştirmesin)
cw = np.clip(cw, 0.1, 10.0)
class_weight_dict = {i: cw[i] for i in range(NUM_CLASSES)}

print(f"Class weight — min: {cw.min():.3f}, max: {cw.max():.3f}, mean: {cw.mean():.3f}")

In [ ]:
# ── COMPILE ───────────────────────────────────────────────
# Cosine decay with warmup
steps_per_epoch = len(X_train) // BATCH_SIZE
total_steps     = steps_per_epoch * EPOCHS
warmup_steps    = steps_per_epoch * 5  # 5 epoch warmup

lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3,
    decay_steps=total_steps - warmup_steps,
    alpha=1e-5
)

optimizer = keras.optimizers.AdamW(
    learning_rate=lr_schedule,
    weight_decay=1e-4
)

model.compile(
    optimizer=optimizer,
    loss=keras.losses.SparseCategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

print("Model compile edildi")

In [ ]:
# ── CALLBACKS ─────────────────────────────────────────────
callbacks = [
    keras.callbacks.ModelCheckpoint(
        SAVE_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.CSVLogger(f"{BASE}/training_log.csv"),
]

print(f"Model kaydedilecek yer: {SAVE_PATH}")

In [ ]:
# ── EĞİTİM ────────────────────────────────────────────────
print("Eğitim başlıyor...")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Max epochs: {EPOCHS}")
print(f"  Early stopping patience: {PATIENCE}")

start = time.time()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)

elapsed = time.time() - start
print(f"\nEğitim tamamlandı: {elapsed/60:.1f} dakika")

# History kaydet
with open(HISTORY_PATH, 'w') as f:
    json.dump({
        'accuracy':     history.history.get('accuracy', []),
        'val_accuracy': history.history.get('val_accuracy', []),
        'loss':         history.history.get('loss', []),
        'val_loss':     history.history.get('val_loss', [])
    }, f)

In [ ]:
# ── DEĞERLENDİRME ─────────────────────────────────────────
print("\n" + "="*50)
print("SONUÇLAR")
print("="*50)

# En iyi model yükle
best_model = keras.models.load_model(SAVE_PATH)

# Val üzerinde tahmin
y_pred_proba = best_model.predict(val_ds, verbose=1)

# Top-1
top1 = top_k_accuracy_score(y_val, y_pred_proba, k=1)
top3 = top_k_accuracy_score(y_val, y_pred_proba, k=3)
top5 = top_k_accuracy_score(y_val, y_pred_proba, k=5)

print(f"Top-1 Accuracy: {top1:.4f} ({top1*100:.2f}%)")
print(f"Top-3 Accuracy: {top3:.4f} ({top3*100:.2f}%)")
print(f"Top-5 Accuracy: {top5:.4f} ({top5*100:.2f}%)")

In [ ]:
# ── CLASS BAZLI ANALİZ ────────────────────────────────────
sign_df = pd.read_csv(SIGN_CSV)
label_map = dict(zip(sign_df['ClassId'], sign_df['TR']))

y_pred = np.argmax(y_pred_proba, axis=1)

per_class = []
for c in range(NUM_CLASSES):
    mask = y_val == c
    if mask.sum() == 0:
        continue
    acc = (y_pred[mask] == c).mean()
    per_class.append({
        'class_id': c,
        'TR': label_map.get(c, '?'),
        'n_val': int(mask.sum()),
        'accuracy': float(acc)
    })

per_class_df = pd.DataFrame(per_class).sort_values('accuracy', ascending=False)
per_class_df.to_csv(f"{BASE}/per_class_accuracy.csv", index=False)

print("\nEn iyi 20 sınıf:")
print(per_class_df.head(20).to_string(index=False))

print("\nEn kötü 20 sınıf:")
print(per_class_df.tail(20).to_string(index=False))

print(f"\n✅ per_class_accuracy.csv kaydedildi")

In [ ]:
# ── DEMO BİLGİLERİNİ KAYDET ───────────────────────────────
# Web demo'nun ihtiyaç duyduğu dosyalar
DEMO_DIR = f"{BASE}/demo_assets"
os.makedirs(DEMO_DIR, exist_ok=True)

# 1) norm_stats.json zaten kaydedildi
import shutil
shutil.copy(f"{BASE}/norm_stats.json", f"{DEMO_DIR}/norm_stats.json")

# 2) label_map.json — class_id → {TR, EN}
label_map_full = {}
for _, row in sign_df.iterrows():
    label_map_full[int(row['ClassId'])] = {
        'TR': str(row['TR']),
        'EN': str(row['EN'])
    }
with open(f"{DEMO_DIR}/label_map.json", 'w', encoding='utf-8') as f:
    json.dump(label_map_full, f, ensure_ascii=False, indent=2)

# 3) Model kopyala
shutil.copy(SAVE_PATH, f"{DEMO_DIR}/model.keras")

# 4) Demo config
demo_config = {
    'seq_len': 16,
    'feat_dim': 450,
    'num_classes': NUM_CLASSES,
    'landmark_layout': 'color_left_hand(63)+color_right_hand(63)+color_pose(99)+depth_left_hand(63)+depth_right_hand(63)+depth_pose(99)',
    'model_file': 'model.keras',
    'norm_stats_file': 'norm_stats.json',
    'label_map_file': 'label_map.json',
    'confidence_threshold': 0.4,
    'top_k_display': 3
}
with open(f"{DEMO_DIR}/demo_config.json", 'w') as f:
    json.dump(demo_config, f, indent=2)

print(f"\n✅ Demo dosyaları kaydedildi: {DEMO_DIR}")
print("  - model.keras")
print("  - norm_stats.json")
print("  - label_map.json")
print("  - demo_config.json")
print("\nBu 4 dosyayı bilgisayarına indirip web demo klasörüne koy.")